# ACD example: ADHDP (`design="adhdp"`) on ImprovedB747Env

This notebook demonstrates `tensoraerospace.agent.ADP` in **ADHDP** mode.

- Critic: learns action-dependent cost-to-go \(Q(s,a)\) via TD.
- Actor: deterministic policy updated to minimize \(Q(s,\pi(s))\).

This is the *action-dependent* scalar critic family (closest to ADHDP/HDP ideas, but implemented in a practical modern TD way).



In [ ]:
import sys
from pathlib import Path

# Make sure repo root + example helpers are importable
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "tensoraerospace").exists() else HERE.parents[2]
EX_DIR = ROOT / "example" / "dynamic-programming"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(EX_DIR))

import numpy as np

from tensoraerospace.agent import ADP
from acd_b747_common import make_env_b747_sine, plot_rollout, rollout

print("repo root:", ROOT)



In [ ]:
# ---- User controls ----
DT = 0.1
N_STEPS = 300
REWARD_MODE = "tracking"  # "tracking" or "step_response"

# Sine reference (pitch)
SINE_AMP_DEG = 1.0
SINE_FREQ_HZ = 0.05

# ADP/ACD
DESIGN = "adhdp"
DEVICE = "cpu"
GAMMA = 0.99
HIDDEN_SIZE = 64

# Training
TRAIN_EPISODES = 200
MAX_STEPS_PER_EPISODE = N_STEPS

# Practical stabilizers (optional): replay + target networks
USE_REPLAY = True
REPLAY_CAPACITY = 50_000
BATCH_SIZE = 64
UPDATES_PER_STEP = 1

USE_TARGET_NETWORKS = True
TAU = 0.02

ACTOR_LR = 3e-4
CRITIC_LR = 3e-4
EXPLORATION_STD = 0.05



In [ ]:
def make_env():
    return make_env_b747_sine(
        dt=DT,
        n_steps=N_STEPS,
        reward_mode=REWARD_MODE,
        sine_amp_deg=SINE_AMP_DEG,
        sine_freq_hz=SINE_FREQ_HZ,
    )

# Baseline (random)
baseline = rollout(make_env(), agent=None, baseline="random")
plot_rollout(baseline, title="Baseline (random)")

agent = ADP(
    env=make_env(),
    design=DESIGN,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    hidden_size=HIDDEN_SIZE,
    device=DEVICE,
    exploration_std=EXPLORATION_STD,
    use_replay=USE_REPLAY,
    memory_capacity=REPLAY_CAPACITY,
    batch_size=BATCH_SIZE,
    updates_per_step=UPDATES_PER_STEP,
    use_target_networks=USE_TARGET_NETWORKS,
    tau=TAU,
    log_every_updates=500,
)

print("Training...", "episodes=", TRAIN_EPISODES, "design=", DESIGN)
agent.train(num_episodes=int(TRAIN_EPISODES), max_steps=int(MAX_STEPS_PER_EPISODE))

trained = rollout(make_env(), agent=agent, deterministic=True)
plot_rollout(trained, title=f"Trained ({DESIGN})")

